# 🎯 Retrieval Strategies: Finding the Right Information

Welcome to **Notebook 04** in our RAG series! Now that you understand embeddings (Notebook 01) and vector stores (Notebook 03), it's time to learn **how to actually find the right information** when someone asks a question.

---

### 📚 What You'll Learn

| Topic | What It Means |
|-------|---------------|
| **Dense Retrieval** | Finding documents by *meaning* using embeddings |
| **Sparse Retrieval (BM25 / TF-IDF)** | Finding documents by *exact keywords* |
| **Hybrid Retrieval** | Combining both approaches for better results |
| **Re-ranking** | Polishing your results to get the very best ones on top |
| **Choosing the Right Strategy** | When to use which approach |

### 📋 Prerequisites

- **Notebook 01** — Understanding Embeddings
- **Notebook 03** — Vector Stores & Indexing

### 🛠️ Requirements

This notebook uses **only built-in Python libraries**: `numpy`, `matplotlib`, `math`, `re`, `collections`. No external packages needed!

Let's dive in! 🏊

---

## 🏛️ The Retrieval Challenge

Imagine you walk into a **massive library** with millions of books. Someone asks you a question, and you need to find the best books to answer it. How do you search?

### 📖 Three Ways to Find a Book

**Method 1: Search by Exact Title / Keywords → Sparse Retrieval** 🔍
> You go to the card catalog and look up the *exact words* from the question. If someone asks about "chocolate cake recipes," you search for books with those exact words in the title or description. Fast and precise — but you'll miss a book called "Delicious Cocoa Desserts" even though it's perfect!

**Method 2: Describe What It's ABOUT → Dense Retrieval** 🧠
> You talk to a librarian who *understands meaning*. You say "I want something about making sweet baked treats with chocolate." The librarian knows that "cocoa desserts" and "chocolate cake" are about the same thing. Flexible and smart — but sometimes the librarian overthinks it and misses an obvious keyword match!

**Method 3: Use BOTH Methods → Hybrid Retrieval** 🤝
> You check the card catalog AND ask the librarian, then combine both recommendations. Now you get the best of both worlds!

**Final Step: Pick the BEST Ones → Re-ranking** 🏆
> You've got a big stack of books. Now you quickly skim each one and re-order them so the absolute best answers are on top. This is re-ranking — a final quality check.

Let's see this visually! 👇

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
fig.suptitle('🎯 Overview: Retrieval Strategies', fontsize=16, fontweight='bold', y=0.98)

# --- Query box (left) ---
query_box = mpatches.FancyBboxPatch((0.3, 1.8), 2.0, 1.4, boxstyle='round,pad=0.15',
                                     facecolor='#FFD700', edgecolor='#B8860B', linewidth=2)
ax.add_patch(query_box)
ax.text(1.3, 2.5, '❓ Query', ha='center', va='center', fontsize=13, fontweight='bold')

# --- Three retrieval boxes (middle) ---
colors = ['#87CEEB', '#98FB98', '#DDA0DD']
labels = ['📝 Sparse\n(Keywords)', '🧠 Dense\n(Meaning)', '🤝 Hybrid\n(Both)']
y_positions = [3.6, 1.8, 0.0]

for i, (color, label, y) in enumerate(zip(colors, labels, y_positions)):
    box = mpatches.FancyBboxPatch((4.5, y + 0.1), 2.4, 1.2, boxstyle='round,pad=0.15',
                                  facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    ax.text(5.7, y + 0.7, label, ha='center', va='center', fontsize=11, fontweight='bold')
    # Arrow from query to each box
    ax.annotate('', xy=(4.5, y + 0.7), xytext=(2.3, 2.5),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# --- Re-ranker box (right-middle) ---
rerank_box = mpatches.FancyBboxPatch((9.0, 1.5), 2.2, 1.8, boxstyle='round,pad=0.15',
                                      facecolor='#FF6347', edgecolor='#B22222', linewidth=2)
ax.add_patch(rerank_box)
ax.text(10.1, 2.4, '🏆 Re-ranker\n(Polish)', ha='center', va='center',
        fontsize=12, fontweight='bold', color='white')

# Arrows from each retrieval to re-ranker
for y in y_positions:
    ax.annotate('', xy=(9.0, 2.4), xytext=(6.9, y + 0.7),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# --- Final results box ---
result_box = mpatches.FancyBboxPatch((12.0, 1.8), 1.7, 1.4, boxstyle='round,pad=0.15',
                                      facecolor='#32CD32', edgecolor='#228B22', linewidth=2)
ax.add_patch(result_box)
ax.text(12.85, 2.5, '✅ Top\nResults', ha='center', va='center',
        fontsize=12, fontweight='bold', color='white')
ax.annotate('', xy=(12.0, 2.5), xytext=(11.2, 2.4),
            arrowprops=dict(arrowstyle='->', color='#555', lw=2))

plt.tight_layout()
plt.show()

---

## 📝 Sparse Retrieval — The Keyword Approach

Sparse retrieval works by matching **exact keywords** between the query and documents. Think of it like a **word-counting detective** 🕵️ — it counts how important each word is.

### 🔢 TF-IDF (Term Frequency × Inverse Document Frequency)

TF-IDF is a simple but powerful idea built from two ingredients:

| Component | What It Measures | Analogy |
|-----------|-----------------|--------|
| **TF** (Term Frequency) | How often a word appears in **THIS** document | If a book mentions "dogs" 50 times, it's probably about dogs! |
| **IDF** (Inverse Document Frequency) | How **rare** a word is across **ALL** documents | "The" appears everywhere (boring!). "Quantum" is rare (interesting!) |

**TF-IDF = TF × IDF**

A word scores HIGH when it appears **often in this document** but **rarely in others**. That makes it a great identifier for this document!

### 🚀 BM25: The Improved Version

BM25 (Best Matching 25) is like TF-IDF's **smarter older sibling**. It adds two improvements:

1. **Diminishing returns** 📉 — If a word appears 100 times vs 10 times, it doesn't get 10× the score. After a while, more occurrences barely matter.
2. **Length normalization** 📏 — A short document mentioning "dogs" 5 times is more focused than a long book mentioning it 5 times.

BM25 is considered the **"gold standard"** of keyword-based search. Most search engines (like Elasticsearch) use it as their default!

Let's implement both from scratch 👇

In [ ]:
import math
import re
from collections import Counter

# ============================================================
# 📄 Sample document collection
# ============================================================
documents = [
    "Cats are independent pets that love to chase mice and play with yarn",
    "Dogs are loyal animals that enjoy fetching balls and going for walks",
    "Machine learning algorithms can recognize cats and dogs in photos",
    "Python programming is popular for building web applications and data science",
    "Cooking pasta requires boiling water and adding the right amount of salt",
    "Basketball and soccer are popular team sports played around the world",
    "Deep learning neural networks are transforming computer vision technology",
    "Italian cuisine features pasta, pizza, and fresh Mediterranean ingredients",
]

doc_labels = [
    "🐱 Cats", "🐶 Dogs", "🤖 ML + Animals", "💻 Python",
    "🍝 Cooking Pasta", "⚽ Sports", "🧠 Deep Learning", "🇮🇹 Italian Food"
]


def tokenize(text):
    """Split text into lowercase words, removing punctuation."""
    return re.findall(r'[a-z]+', text.lower())


# ============================================================
# 📊 TF-IDF Implementation
# ============================================================

def compute_tf(tokens):
    """Term Frequency: count of each word / total words in doc."""
    counts = Counter(tokens)
    total = len(tokens)
    return {word: count / total for word, count in counts.items()}


def compute_idf(corpus_tokens):
    """Inverse Document Frequency: log(N / docs containing word)."""
    n_docs = len(corpus_tokens)
    # Count how many documents contain each word
    doc_freq = Counter()
    for tokens in corpus_tokens:
        unique_words = set(tokens)
        for word in unique_words:
            doc_freq[word] += 1
    # IDF = log(N / df)  (with +1 smoothing to avoid division by zero)
    return {word: math.log((n_docs + 1) / (df + 1)) + 1
            for word, df in doc_freq.items()}


def tfidf_search(query, documents, top_k=5):
    """Search documents using TF-IDF scoring."""
    corpus_tokens = [tokenize(doc) for doc in documents]
    idf = compute_idf(corpus_tokens)
    query_tokens = tokenize(query)

    scores = []
    for i, doc_tokens in enumerate(corpus_tokens):
        tf = compute_tf(doc_tokens)
        score = 0.0
        for qt in query_tokens:
            if qt in tf and qt in idf:
                score += tf[qt] * idf[qt]
        scores.append((i, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


# ============================================================
# 🧪 Demo: TF-IDF in action
# ============================================================
query = "cats and dogs in machine learning"
print("=" * 60)
print(f"🔍 Query: \"{query}\"")
print("=" * 60)

# Show step-by-step reasoning
query_tokens = tokenize(query)
corpus_tokens = [tokenize(doc) for doc in documents]
idf = compute_idf(corpus_tokens)

print(f"\n📝 Query tokens: {query_tokens}")
print(f"\n📊 IDF values for query words:")
for qt in query_tokens:
    idf_val = idf.get(qt, 0)
    print(f"   '{qt}': IDF = {idf_val:.3f}" +
          (" (common word — low IDF)" if idf_val < 1.5 else
           " (somewhat rare)" if idf_val < 2.0 else " (rare word — high IDF)"))

print(f"\n🏆 TF-IDF Search Results:")
print("-" * 50)
results = tfidf_search(query, documents)
for rank, (idx, score) in enumerate(results, 1):
    if score > 0:
        print(f"  #{rank}  Score: {score:.4f}  {doc_labels[idx]}")
        print(f"       \"{documents[idx]}\"")

In [ ]:
# ============================================================
# 🚀 BM25 Implementation
# ============================================================

def bm25_search(query, documents, k1=1.5, b=0.75, top_k=5):
    """
    BM25 search — the gold standard of keyword retrieval.

    Parameters:
        k1: Controls diminishing returns of term frequency (1.2–2.0 typical)
        b:  Controls length normalization (0 = none, 1 = full). 0.75 is standard.
    """
    corpus_tokens = [tokenize(doc) for doc in documents]
    n_docs = len(documents)

    # Average document length
    avg_dl = sum(len(t) for t in corpus_tokens) / n_docs

    # Document frequency for each word
    doc_freq = Counter()
    for tokens in corpus_tokens:
        for word in set(tokens):
            doc_freq[word] += 1

    query_tokens = tokenize(query)
    scores = []

    for i, doc_tokens in enumerate(corpus_tokens):
        doc_len = len(doc_tokens)
        tf_counts = Counter(doc_tokens)
        score = 0.0

        for qt in query_tokens:
            if qt not in tf_counts:
                continue
            tf = tf_counts[qt]
            df = doc_freq.get(qt, 0)

            # IDF component (with smoothing)
            idf = math.log((n_docs - df + 0.5) / (df + 0.5) + 1)

            # BM25 TF component: diminishing returns + length normalization
            tf_component = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_dl))

            score += idf * tf_component

        scores.append((i, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


# ============================================================
# 🧪 Demo: BM25 in action
# ============================================================
print("=" * 60)
print(f"🔍 Query: \"{query}\"")
print("=" * 60)

print(f"\n🚀 BM25 Search Results:")
print("-" * 50)
bm25_results = bm25_search(query, documents)
for rank, (idx, score) in enumerate(bm25_results, 1):
    if score > 0:
        print(f"  #{rank}  Score: {score:.4f}  {doc_labels[idx]}")
        print(f"       \"{documents[idx]}\"")

# ============================================================
# 📊 Compare TF-IDF vs BM25
# ============================================================
print("\n" + "=" * 60)
print("📊 Comparison: TF-IDF vs BM25")
print("=" * 60)

tfidf_results = tfidf_search(query, documents, top_k=8)
bm25_results_full = bm25_search(query, documents, top_k=8)

tfidf_dict = {idx: score for idx, score in tfidf_results}
bm25_dict = {idx: score for idx, score in bm25_results_full}

print(f"{'Document':<25} {'TF-IDF':>10} {'BM25':>10}")
print("-" * 50)
for i in range(len(documents)):
    tf_score = tfidf_dict.get(i, 0)
    bm_score = bm25_dict.get(i, 0)
    if tf_score > 0 or bm_score > 0:
        print(f"{doc_labels[i]:<25} {tf_score:>10.4f} {bm_score:>10.4f}")

---

## 🧠 Dense Retrieval — The Meaning Approach

Dense retrieval doesn't count keywords — it understands **meaning**.

### How It Works

1. Every document is converted into a **vector** (a list of numbers) called an **embedding**
2. The query is also converted into a vector
3. We find documents whose vectors are **closest** to the query vector

### 🌟 The Magic: Understanding Meaning

| Query | Keyword Search Finds | Dense Search Finds |
|-------|---------------------|-------------------|
| "What do cats eat?" | Docs with "cats" and "eat" | "Feline dietary requirements" ✅ |
| "automobile speed" | Docs with "automobile" | "How fast can cars go?" ✅ |
| "happy" | Docs with "happy" | "joyful", "cheerful", "delighted" ✅ |

### ✅ Pros
- Handles **synonyms** (car = automobile)
- Understands **paraphrases** ("How old are you?" ≈ "What is your age?")
- Can work **cross-language** (with multilingual models)

### ❌ Cons
- Needs an **embedding model** (adds complexity)
- Can sometimes **miss exact keyword matches** that are obviously relevant
- Harder to **debug** (why did it pick this document?)

Let's build a simplified version 👇

In [ ]:
import numpy as np

# ============================================================
# 🧠 Dense Retrieval (Simplified with Word-Frequency Vectors)
# ============================================================
# In production you'd use a real embedding model (e.g., sentence-transformers).
# Here we simulate the concept using word-frequency vectors + cosine similarity.
# We add a synonym map to illustrate the "meaning" advantage.

# Synonym map — simulates what a real embedding model learns
SYNONYMS = {
    'feline': ['cat', 'cats'],
    'canine': ['dog', 'dogs'],
    'automobile': ['car', 'cars'],
    'cuisine': ['cooking', 'food', 'cook'],
    'athletic': ['sports', 'sport'],
    'ai': ['machine', 'learning', 'neural', 'deep'],
    'programming': ['python', 'code', 'coding'],
    'pasta': ['spaghetti', 'noodles'],
}


def expand_with_synonyms(tokens):
    """Expand tokens with synonyms to simulate semantic understanding."""
    expanded = list(tokens)
    for token in tokens:
        # Direct synonym lookup
        if token in SYNONYMS:
            expanded.extend(SYNONYMS[token])
        # Reverse lookup: if token is a synonym value, add the key & siblings
        for key, values in SYNONYMS.items():
            if token in values:
                expanded.append(key)
                expanded.extend(v for v in values if v != token)
    return expanded


def build_vocab(corpus_tokens):
    """Build vocabulary from all documents."""
    vocab = set()
    for tokens in corpus_tokens:
        expanded = expand_with_synonyms(tokens)
        vocab.update(expanded)
    return sorted(vocab)


def to_vector(tokens, vocab):
    """Convert tokens to a frequency vector over the vocabulary."""
    expanded = expand_with_synonyms(tokens)
    counts = Counter(expanded)
    return np.array([counts.get(w, 0) for w in vocab], dtype=float)


def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return dot / norm


def dense_search(query, documents, top_k=5):
    """Dense (semantic) search using word-frequency vectors + synonyms."""
    corpus_tokens = [tokenize(doc) for doc in documents]
    query_tokens = tokenize(query)

    # Build shared vocabulary
    all_tokens = corpus_tokens + [query_tokens]
    vocab = build_vocab(all_tokens)

    # Convert to vectors
    query_vec = to_vector(query_tokens, vocab)
    doc_vecs = [to_vector(t, vocab) for t in corpus_tokens]

    # Compute similarities
    scores = []
    for i, dv in enumerate(doc_vecs):
        sim = cosine_similarity(query_vec, dv)
        scores.append((i, sim))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]


# ============================================================
# 🧪 Demo: Dense retrieval finds what keywords miss!
# ============================================================
semantic_query = "feline and canine recognition using AI"

print("=" * 65)
print(f'🔍 Query: "{semantic_query}"')
print("=" * 65)
print("\nNotice: the query says 'feline' (not 'cat') and 'canine' (not 'dog')!")
print("Keyword search will struggle, but dense search understands synonyms.\n")

# Keyword search (BM25)
print("📝 BM25 (Keyword) Results:")
print("-" * 50)
kw_results = bm25_search(semantic_query, documents)
found_any = False
for rank, (idx, score) in enumerate(kw_results, 1):
    if score > 0:
        found_any = True
        print(f"  #{rank}  Score: {score:.4f}  {doc_labels[idx]}")
if not found_any:
    print("  ❌ No results! Keywords don't match any documents.")

# Dense search
print(f"\n🧠 Dense (Semantic) Results:")
print("-" * 50)
dense_results = dense_search(semantic_query, documents)
for rank, (idx, score) in enumerate(dense_results, 1):
    if score > 0.05:
        print(f"  #{rank}  Score: {score:.4f}  {doc_labels[idx]}")
        print(f"       \"{documents[idx]}\"")

print("\n💡 Dense retrieval found the right documents because it understands")
print("   that 'feline' ≈ 'cat' and 'canine' ≈ 'dog'!")

In [ ]:
# ============================================================
# 📊 Side-by-side comparison: Sparse vs Dense
# ============================================================

comparison_query = "feline and canine recognition using AI"

sparse_results = bm25_search(comparison_query, documents, top_k=8)
dense_results_all = dense_search(comparison_query, documents, top_k=8)

sparse_scores = {idx: score for idx, score in sparse_results}
dense_scores = {idx: score for idx, score in dense_results_all}

# Normalize scores to [0, 1] for fair comparison
max_sparse = max((s for s in sparse_scores.values()), default=1) or 1
max_dense = max((s for s in dense_scores.values()), default=1) or 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'📊 Sparse vs Dense Retrieval\nQuery: "{comparison_query}"',
             fontsize=14, fontweight='bold')

y_pos = np.arange(len(documents))

# Sparse (BM25) results
sparse_vals = [sparse_scores.get(i, 0) / max_sparse for i in range(len(documents))]
bars1 = ax1.barh(y_pos, sparse_vals, color='#87CEEB', edgecolor='#4682B4', linewidth=1.2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(doc_labels, fontsize=10)
ax1.set_xlabel('Normalized Score', fontsize=11)
ax1.set_title('📝 Sparse (BM25) — Keywords', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 1.1)
ax1.invert_yaxis()
for bar, val in zip(bars1, sparse_vals):
    if val > 0:
        ax1.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=9)

# Dense results
dense_vals = [dense_scores.get(i, 0) / max_dense for i in range(len(documents))]
bars2 = ax2.barh(y_pos, dense_vals, color='#98FB98', edgecolor='#2E8B57', linewidth=1.2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(doc_labels, fontsize=10)
ax2.set_xlabel('Normalized Score', fontsize=11)
ax2.set_title('🧠 Dense — Meaning', fontsize=13, fontweight='bold')
ax2.set_xlim(0, 1.1)
ax2.invert_yaxis()
for bar, val in zip(bars2, dense_vals):
    if val > 0.01:
        ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 Notice how sparse search may score 0 for synonym queries,")
print("   while dense search successfully finds relevant documents!")

---

## 🤝 Hybrid Retrieval — Best of Both Worlds

Why choose between keywords and meaning when you can have **both**? Hybrid retrieval combines sparse and dense search to cover each other's weaknesses.

### 🔧 Method 1: Weighted Combination

The simplest approach — just add the scores together with a weight:

```
hybrid_score = alpha × sparse_score + (1 - alpha) × dense_score
```

- `alpha = 1.0` → pure sparse (keywords only)
- `alpha = 0.0` → pure dense (meaning only)
- `alpha = 0.5` → equal mix
- `alpha = 0.3` → lean toward meaning (common in practice)

### 🔧 Method 2: Reciprocal Rank Fusion (RRF)

A smarter approach that uses **ranks** instead of raw scores:

```
RRF_score = 1/(k + rank_sparse) + 1/(k + rank_dense)
```

Why is this better?
- Scores from different methods aren't comparable (BM25 might give 5.2 while cosine gives 0.8)
- Ranks are always on the same scale (1st, 2nd, 3rd...)
- RRF is **more robust** and used in production systems

The constant `k` (typically 60) controls how much we favor top-ranked documents.

Let's implement both! 👇

In [ ]:
# ============================================================
# 🤝 Hybrid Retrieval: Weighted + RRF
# ============================================================

def normalize_scores(results, n_docs):
    """Normalize scores to [0, 1] range."""
    scores = {idx: score for idx, score in results}
    max_score = max(scores.values()) if scores else 1
    if max_score == 0:
        max_score = 1
    return {idx: scores.get(idx, 0) / max_score for idx in range(n_docs)}


def hybrid_search(query, documents, alpha=0.5, top_k=5):
    """
    Hybrid search: weighted combination of sparse (BM25) + dense.
    alpha = weight for sparse; (1 - alpha) = weight for dense.
    """
    n = len(documents)
    sparse = normalize_scores(bm25_search(query, documents, top_k=n), n)
    dense = normalize_scores(dense_search(query, documents, top_k=n), n)

    combined = []
    for i in range(n):
        score = alpha * sparse[i] + (1 - alpha) * dense[i]
        combined.append((i, score))

    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:top_k]


def rrf_search(query, documents, k=60, top_k=5):
    """
    Reciprocal Rank Fusion: combine rankings from multiple methods.
    RRF_score(d) = sum over methods: 1 / (k + rank)
    """
    n = len(documents)

    # Get ranked lists
    sparse_ranked = bm25_search(query, documents, top_k=n)
    dense_ranked = dense_search(query, documents, top_k=n)

    # Assign ranks (1-indexed)
    sparse_ranks = {idx: rank for rank, (idx, _) in enumerate(sparse_ranked, 1)}
    dense_ranks = {idx: rank for rank, (idx, _) in enumerate(dense_ranked, 1)}

    # Compute RRF scores
    rrf_scores = []
    for i in range(n):
        sr = sparse_ranks.get(i, n + 1)
        dr = dense_ranks.get(i, n + 1)
        score = 1.0 / (k + sr) + 1.0 / (k + dr)
        rrf_scores.append((i, score))

    rrf_scores.sort(key=lambda x: x[1], reverse=True)
    return rrf_scores[:top_k]


# ============================================================
# 🧪 Demo: Hybrid finds what both methods individually miss
# ============================================================
# Use a query where sparse finds one thing and dense finds another
hybrid_query = "feline pets and pasta cooking"

print("=" * 65)
print(f'🔍 Query: "{hybrid_query}"')
print("=" * 65)
print("\nThis query has:")
print('  • "feline" → Dense should find cat docs (synonym)' )
print('  • "pasta cooking" → Sparse should find cooking/Italian docs (exact keyword)')
print('  • Hybrid should find BOTH!\n')

n = len(documents)

# Sparse only
print("📝 BM25 (Sparse) Top 3:")
for rank, (idx, score) in enumerate(bm25_search(hybrid_query, documents, top_k=3), 1):
    if score > 0:
        print(f"  #{rank} {doc_labels[idx]:20s} (score: {score:.4f})")

# Dense only
print("\n🧠 Dense Top 3:")
for rank, (idx, score) in enumerate(dense_search(hybrid_query, documents, top_k=3), 1):
    if score > 0.05:
        print(f"  #{rank} {doc_labels[idx]:20s} (score: {score:.4f})")

# Hybrid (weighted)
print("\n🤝 Hybrid (Weighted, alpha=0.5) Top 3:")
for rank, (idx, score) in enumerate(hybrid_search(hybrid_query, documents, alpha=0.5, top_k=3), 1):
    if score > 0:
        print(f"  #{rank} {doc_labels[idx]:20s} (score: {score:.4f})")

# RRF
print("\n🔀 Reciprocal Rank Fusion (RRF) Top 3:")
for rank, (idx, score) in enumerate(rrf_search(hybrid_query, documents, top_k=3), 1):
    print(f"  #{rank} {doc_labels[idx]:20s} (score: {score:.6f})")

print("\n✅ Hybrid methods surface documents from BOTH sparse and dense results!")

In [ ]:
# ============================================================
# 📊 Grouped bar chart: Sparse vs Dense vs Hybrid
# ============================================================

chart_query = "feline pets and pasta cooking"
n = len(documents)

# Get normalized scores for all three methods
sp = normalize_scores(bm25_search(chart_query, documents, top_k=n), n)
dn = normalize_scores(dense_search(chart_query, documents, top_k=n), n)
hy = normalize_scores(hybrid_search(chart_query, documents, alpha=0.5, top_k=n), n)

sp_vals = [sp[i] for i in range(n)]
dn_vals = [dn[i] for i in range(n)]
hy_vals = [hy[i] for i in range(n)]

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(n)
width = 0.25

bars_sp = ax.bar(x - width, sp_vals, width, label='📝 Sparse (BM25)',
                  color='#87CEEB', edgecolor='#4682B4', linewidth=1)
bars_dn = ax.bar(x, dn_vals, width, label='🧠 Dense',
                  color='#98FB98', edgecolor='#2E8B57', linewidth=1)
bars_hy = ax.bar(x + width, hy_vals, width, label='🤝 Hybrid',
                  color='#DDA0DD', edgecolor='#8B008B', linewidth=1)

# Highlight the winner for each document
for i in range(n):
    vals = [sp_vals[i], dn_vals[i], hy_vals[i]]
    max_val = max(vals)
    if max_val > 0.05:
        winner_x = [x[i] - width, x[i], x[i] + width][vals.index(max_val)]
        ax.annotate('★', xy=(winner_x, max_val + 0.02), ha='center',
                    fontsize=12, color='gold')

ax.set_xlabel('Documents', fontsize=12)
ax.set_ylabel('Normalized Score', fontsize=12)
ax.set_title(f'📊 Score Comparison: Sparse vs Dense vs Hybrid\nQuery: "{chart_query}"',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(doc_labels, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("★ = highest scoring method for that document")
print("\n💡 Notice how Hybrid captures relevant docs from BOTH methods!")

---

## 🏆 Re-ranking — Polishing Your Results

After retrieval, we have a list of *possibly* relevant documents. But are they in the **best** order? Re-ranking is the final step that makes sure the truly best answers bubble to the top.

### 🎭 The Interview Analogy

Think of hiring:
1. **Resume screening** (retrieval) — quickly filter 10,000 applicants → 50 candidates. Fast but rough.
2. **Final interview with the CEO** (re-ranking) — carefully evaluate 50 → hire the best 5. Slow but accurate.

You can't interview 10,000 people (too slow!), but you CAN carefully evaluate 50.

### 🔬 Cross-Encoder vs Bi-Encoder

| Approach | How It Works | Speed | Quality |
|----------|-------------|-------|---------|
| **Bi-encoder** | Embeds query and document *separately*, compares vectors | ⚡ Fast | Good |
| **Cross-encoder** | Processes query AND document *together* as one input | 🐌 Slow | Excellent |

The cross-encoder is too slow for initial retrieval (can't compare against millions of docs), but it's perfect for re-ranking just 50-100 candidates!

### 📋 The Re-ranking Pipeline

```
All Documents (10,000+)
    ↓ Initial Retrieval (fast, rough)
Candidates (top 50)
    ↓ Re-ranking (slow, precise)
Final Results (top 5)
```

Let's visualize this pipeline! 👇

In [ ]:
# ============================================================
# 🏆 Funnel Diagram: The Re-ranking Pipeline
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('🏆 The Re-ranking Pipeline (Funnel)', fontsize=16, fontweight='bold')

# Funnel layers (from top to bottom): wider → narrower
layers = [
    {'y': 7.8, 'width': 8.0, 'height': 1.6, 'color': '#87CEEB',
     'edge': '#4682B4', 'label': '📚 All Documents', 'count': '10,000+',
     'desc': 'Your entire knowledge base'},
    {'y': 5.0, 'width': 5.5, 'height': 1.6, 'color': '#FFD700',
     'edge': '#B8860B', 'label': '🔍 Initial Retrieval', 'count': 'Top 50',
     'desc': 'BM25 / Dense / Hybrid (fast)'},
    {'y': 2.2, 'width': 3.0, 'height': 1.6, 'color': '#32CD32',
     'edge': '#228B22', 'label': '🏆 After Re-ranking', 'count': 'Top 5',
     'desc': 'Cross-encoder (precise)'},
]

for layer in layers:
    x_center = 5.0
    x_left = x_center - layer['width'] / 2
    rect = mpatches.FancyBboxPatch(
        (x_left, layer['y']), layer['width'], layer['height'],
        boxstyle='round,pad=0.2', facecolor=layer['color'],
        edgecolor=layer['edge'], linewidth=2.5
    )
    ax.add_patch(rect)

    # Label text
    ax.text(x_center, layer['y'] + layer['height'] * 0.65,
            f"{layer['label']}  ({layer['count']})",
            ha='center', va='center', fontsize=13, fontweight='bold')
    ax.text(x_center, layer['y'] + layer['height'] * 0.25,
            layer['desc'], ha='center', va='center', fontsize=10,
            fontstyle='italic', color='#333')

# Arrows between layers
arrow_ys = [(9.4, 7.8), (6.6, 5.0), (3.8, 2.2)]
# We only need arrows between the layers:
ax.annotate('', xy=(5, 7.8), xytext=(5, 6.6),
            arrowprops=dict(arrowstyle='<-', color='#555', lw=2.5, ls='--'))
ax.annotate('', xy=(5, 5.0), xytext=(5, 3.8),
            arrowprops=dict(arrowstyle='<-', color='#555', lw=2.5, ls='--'))

# Side labels for filtering
ax.text(9.3, 7.0, 'Fast filter\n(milliseconds)', ha='center', fontsize=10, color='#4682B4',
        fontweight='bold')
ax.text(9.3, 4.2, 'Precise scoring\n(seconds)', ha='center', fontsize=10, color='#B8860B',
        fontweight='bold')

# Connecting lines to side labels
ax.plot([7.8, 8.8], [7.2, 7.2], color='#aaa', lw=1, ls=':')
ax.plot([7.0, 8.8], [4.4, 4.4], color='#aaa', lw=1, ls=':')

# Bottom note
ax.text(5, 0.8, '✅ Only the most relevant documents reach the user!',
        ha='center', va='center', fontsize=12, fontweight='bold',
        color='#228B22',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0fff0', edgecolor='#228B22'))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 🏆 Simple Re-ranker Simulation
# ============================================================
# A real re-ranker uses a cross-encoder model. We'll simulate it
# by combining: original score + query coverage + document quality.


def compute_query_coverage(query, document):
    """What fraction of query words appear in the document?"""
    q_tokens = set(tokenize(query))
    d_tokens = set(tokenize(document))
    if not q_tokens:
        return 0.0
    return len(q_tokens & d_tokens) / len(q_tokens)


def compute_doc_quality(document):
    """Simple heuristic: longer, more detailed docs get a small bonus."""
    words = tokenize(document)
    unique_ratio = len(set(words)) / max(len(words), 1)
    # Reward vocabulary diversity (indicates detailed content)
    return min(unique_ratio, 1.0)


def rerank(query, initial_results, documents, top_k=5):
    """
    Re-rank initial retrieval results using multiple signals.

    Combines:
    - Original retrieval score (40%)
    - Query-document coverage (40%)
    - Document quality heuristic (20%)
    """
    # Normalize initial scores
    max_score = max((s for _, s in initial_results), default=1) or 1

    reranked = []
    for idx, orig_score in initial_results:
        norm_score = orig_score / max_score
        coverage = compute_query_coverage(query, documents[idx])
        quality = compute_doc_quality(documents[idx])

        final_score = 0.4 * norm_score + 0.4 * coverage + 0.2 * quality
        reranked.append((idx, final_score, norm_score, coverage, quality))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked[:top_k]


# ============================================================
# 🧪 Demo: Re-ranking changes the order!
# ============================================================
rerank_query = "cats and dogs in machine learning"

print("=" * 65)
print(f'🔍 Query: "{rerank_query}"')
print("=" * 65)

# Initial retrieval
initial = hybrid_search(rerank_query, documents, alpha=0.5, top_k=6)

print("\n📥 BEFORE Re-ranking (Hybrid Retrieval):")
print("-" * 55)
print(f"{'Rank':<6} {'Document':<25} {'Score':>8}")
for rank, (idx, score) in enumerate(initial, 1):
    print(f"  #{rank:<4} {doc_labels[idx]:<25} {score:>8.4f}")

# Re-rank
reranked = rerank(rerank_query, initial, documents, top_k=6)

print("\n📤 AFTER Re-ranking:")
print("-" * 75)
print(f"{'Rank':<6} {'Document':<25} {'Final':>8} {'Orig':>8} {'Cover':>8} {'Quality':>8}")
for rank, (idx, final, orig, cov, qual) in enumerate(reranked, 1):
    print(f"  #{rank:<4} {doc_labels[idx]:<25} {final:>8.4f} {orig:>8.4f} {cov:>8.4f} {qual:>8.4f}")

# Check if order changed
orig_order = [idx for idx, _ in initial]
new_order = [idx for idx, *_ in reranked]
if orig_order != new_order:
    print("\n🔄 Re-ranking changed the order! Better results moved to the top.")
else:
    print("\n✅ Re-ranking confirmed the original order was already good!")

print("\n📊 Score breakdown:")
print("   Final = 0.4 × Original + 0.4 × Query Coverage + 0.2 × Doc Quality")

---

## 🗺️ Choosing the Right Strategy

There's no one-size-fits-all answer. Here's a practical decision guide:

### 🌳 Decision Tree

```
Are you just starting out?
├─ YES → Start with BM25. It's simple, fast, and surprisingly good!
└─ NO
    ├─ Do you need semantic understanding?
    │   (synonyms, paraphrases, meaning-based matching)
    │   ├─ YES → Add Dense Retrieval
    │   └─ NO → BM25 is probably enough
    └─ Building a production system?
        ├─ YES → Hybrid Retrieval + Re-ranking
        │         (This is what most production RAG systems use)
        └─ NO → Pick whichever fits your use case
```

### 💡 Rules of Thumb

| Situation | Recommendation | Why |
|-----------|---------------|-----|
| Small dataset (<1000 docs) | **BM25** | Simple, fast, no model needed |
| Need synonym matching | **Dense** | Understands meaning |
| Technical docs with jargon | **Hybrid** | Jargon needs exact match + context |
| Production RAG system | **Hybrid + Re-ranking** | Best quality |
| Just getting started | **BM25 first** | Add complexity only when needed |

---

## 📊 Strategy Comparison Table

| Strategy | Speed | Quality | Complexity | Best For |
|----------|-------|---------|------------|----------|
| **TF-IDF** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐ | Quick prototyping, simple search |
| **BM25** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | Keyword-heavy domains, starting out |
| **Dense Retrieval** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | Semantic search, synonym handling |
| **Hybrid (Weighted)** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | Balanced needs, most use cases |
| **Hybrid + RRF** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Production systems, robust results |
| **Hybrid + Re-ranking** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Maximum quality, critical applications |

### Reading the table:
- **Speed**: More stars = faster (TF-IDF/BM25 are instant; re-ranking adds latency)
- **Quality**: More stars = better retrieval accuracy
- **Complexity**: More stars = more complex to implement and maintain
- Start simple (top rows) and add complexity only when needed!

---

## ⚠️ Common Misconceptions

**❌ Misconception 1: "Dense retrieval is always better than BM25"**
> **Reality:** BM25 often outperforms dense retrieval on keyword-heavy queries, especially in technical or domain-specific content. When someone searches for "error code 0x80070005", you want exact keyword matching, not semantic understanding!

---

**❌ Misconception 2: "You need expensive embedding models for good retrieval"**
> **Reality:** BM25 requires NO model at all and is surprisingly competitive. Many production systems use BM25 as their primary retrieval method with great results. Start simple!

---

**❌ Misconception 3: "More retrieved documents = better results"**
> **Reality:** Retrieving too many documents adds noise and confuses the LLM. It's better to retrieve 3-5 highly relevant documents than 20 somewhat relevant ones. Quality over quantity!

---

**❌ Misconception 4: "Re-ranking is optional and doesn't matter much"**
> **Reality:** Re-ranking can dramatically improve result quality. Studies show that re-ranking with a cross-encoder can improve retrieval accuracy by 10-30%. It's the difference between "good" and "great" search results.

---

## 🎯 Key Takeaways

Here's what you've learned in this notebook:

- **Sparse retrieval** (TF-IDF, BM25) matches **exact keywords**. Fast, simple, no model needed. BM25 is the gold standard of keyword search.

- **Dense retrieval** uses **embeddings** to match by **meaning**. Handles synonyms and paraphrases, but needs an embedding model.

- **Hybrid retrieval** combines both approaches. Weighted combination is simple; **Reciprocal Rank Fusion (RRF)** is more robust.

- **Re-ranking** is a final polishing step: retrieve many candidates quickly, then re-score with a more powerful model to get the best order.

- **Start with BM25**, then add dense retrieval and re-ranking as needed. Don't over-engineer!

- The **retrieval pipeline** in production: `Query → Hybrid Retrieval (top 50) → Re-ranking (top 5) → LLM`

- No single strategy is always best — the right choice depends on your data, queries, and requirements.

---

## 🤔 Test Your Understanding

Try answering these questions before peeking at the answers!

---

**Question 1:** If a user searches for "automobile maintenance tips" but your documents say "car repair guide", which retrieval method would find the best match?

<details>
<summary>Click to reveal answer</summary>

**Dense retrieval** would work best here because it understands that "automobile" and "car" are synonyms, and "maintenance" and "repair" are related concepts. BM25 would miss this since none of the exact keywords match.

</details>

---

**Question 2:** What's the key difference between TF-IDF and BM25?

<details>
<summary>Click to reveal answer</summary>

BM25 adds two improvements over TF-IDF: (1) **diminishing returns** — more word occurrences give smaller and smaller score increases, and (2) **length normalization** — shorter documents that mention a term get relatively higher scores than long documents. This makes BM25 more robust in practice.

</details>

---

**Question 3:** Why can't we just use a cross-encoder for all retrieval instead of a two-stage pipeline?

<details>
<summary>Click to reveal answer</summary>

A cross-encoder processes the query AND document **together**, which is very slow. With millions of documents, it would take hours to score every single one. Instead, we use fast methods (BM25, bi-encoder) to narrow down to ~50 candidates, then use the cross-encoder only on those 50. This gives us both speed AND quality.

</details>

---

**Question 4:** In Reciprocal Rank Fusion (RRF), why do we use ranks instead of raw scores?

<details>
<summary>Click to reveal answer</summary>

Because raw scores from different methods aren't comparable! BM25 might give a top score of 12.5 while cosine similarity gives 0.92. You can't meaningfully add these together. Ranks, on the other hand, are always on the same scale (1st, 2nd, 3rd...), making them easy to combine fairly.

</details>

---

**Question 5:** You're building a search system for a legal database with very specific terminology. Which strategy would you start with and why?

<details>
<summary>Click to reveal answer</summary>

Start with **BM25** (sparse retrieval) because legal documents have very specific terminology that needs exact matching (e.g., "habeas corpus", "mens rea", specific statute numbers). Then add **dense retrieval** for a hybrid approach to also catch conceptually similar cases. Finally, add **re-ranking** to ensure the most relevant precedents appear first. Legal search is a great example of where hybrid retrieval shines because you need BOTH exact terminology AND semantic understanding.

</details>

---

## 🚀 What's Next?

Now that you understand how to **find** the right information, it's time to put it all together!

In the next notebook, we'll build a **complete RAG pipeline** from scratch:

**[Notebook 05: Building a RAG Pipeline →](./05_building_rag_pipeline.ipynb)**

You'll learn how to:
- Connect retrieval to an LLM
- Handle the full query → retrieve → generate pipeline
- Build a working question-answering system

See you there! 🎉